In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [1]:
import os

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(
    model="gemini-2.0-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
)

In [3]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
from llama_index.core.node_parser import MarkdownElementNodeParser
from llama_cloud_services import LlamaExtract

pdf_file = "../AutoGen_LLM_agent.pdf"
documents = LlamaParse(
    result_type="markdown",
    auto_mode=True,
    auto_mode_trigger_on_table_in_page=True,
).load_data(pdf_file)

Started parsing the file under job_id 91955fd0-fcbe-4807-82d9-5acf591a5747
....

In [5]:
from copy import deepcopy
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex


def get_page_nodes(docs, separator="\n---\n"):
    """Split each document into page node, by separator."""
    nodes = []
    for doc in docs:
        doc_chunks = doc.text.split(separator)
        for doc_chunk in doc_chunks:
            node = TextNode(
                text=doc_chunk,
                metadata=deepcopy(doc.metadata),
            )
            nodes.append(node)

    return nodes

In [8]:
page_nodes = get_page_nodes(documents)
print(type(page_nodes[0].get_content()))
content = page_nodes[0].get_content()
print(content[:500])

<class 'str'>
# AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversations

Qingyun Wu∗

Penn State University

qingyun@autogen.team

Gagan Bansal

Microsoft Research

gaganbansal@microsoft.com

Jieyu Zhang

University of Washington

jieyuz2@cs.washington.edu

Yiran Wu

Penn State University

yiran.wu@psu.edu

Beibin Li

Microsoft Research

beibin.li@microsoft.com

Erkang Zhu

Microsoft Research

erkang.zhu@microsoft.com

Li Jiang

Microsoft

lijiang1@microsoft.com

Xiaoyun Zhang

Microsoft

xi


In [11]:
from llama_index.core.program import LLMTextCompletionProgram

from pydantic import BaseModel, Field
from typing import List, Optional

# Defining the data schema
class Author(BaseModel):
    name: str = Field(description="The name of the author")
    email: str = Field(description="The email of the author")
    company: str = Field(description="The company of the author")

class Authos(BaseModel):
    items: List[Author] = Field(description="The list of authors")


class Paragraph(BaseModel):
    content: str = Field(description="The content of the paragraph")


class Section(BaseModel):
    title: str = Field(description="The title of the section")
    paragraphs: List[Paragraph] = Field(description="The list of paragraphs in the section")

class Paper(BaseModel):
    title: str = Field(description="The title of the paper")
    abstract: str = Field(description="The abstract of the paper")
    authors: Authos = Field(description="The authors of the paper")
    sections: List[Section] = Field(description="The list of sections in the paper")
    


In [12]:
prompt_template_str = """\
From the following information, create the title, abstract, authors and sections of a paper:\
{content} \
"""
program = LLMTextCompletionProgram.from_defaults(
    llm=llm,
    output_cls=Paper,
    prompt_template_str=prompt_template_str,
    verbose=True,
)


output = program(content=content)

In [14]:
output.model_dump()

{'title': 'AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversations',
 'abstract': 'We present AutoGen,1 an open-source framework that allows developers to build LLM applications by composing multiple agents to converse with each other to accomplish tasks. AutoGen agents are customizable, conversable, and can operate in various modes that employ combinations of LLMs, human inputs, and tools. It also enables developers to create flexible agent behaviors and conversation patterns for different applications using both natural language and code. AutoGen serves as a generic infrastructure and is widely used by AI practitioners and researchers to build diverse applications of various complexities and LLM capacities. We demonstrate the framework’s effectiveness with several pilot applications, on domains ranging from mathematics and coding to question-answering, supply-chain optimization, online decision-making, and entertainment.',
 'authors': {'items': [{'name': 'Qingyun Wu